In [0]:
%pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 25.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
%pip install fastapi uvicorn scikit-learn joblib pydantic

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
app = FastAPI()

In [0]:
%pip install fastapi uvicorn nest_asyncio

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =========================================
# IMPORTS
# =========================================
from fastapi import FastAPI, HTTPException
import pandas as pd
import pickle
import uvicorn
from threading import Thread

# =========================================
# INIT APP
# =========================================
app = FastAPI()

# =========================================
# LOAD TABLE 1: ROUTE SEARCH DATA
# =========================================
try:
    route_df = spark.read.table("workspace.default.route_optimisation_data").toPandas()
    print("✓ route_optimisation_data loaded")
except Exception as e:
    print(f"✗ Error loading route_optimisation_data: {e}")
    route_df = pd.DataFrame()

# =========================================
# LOAD TABLE 2: MODEL INPUT DATA
# =========================================
try:
    optimize_df = spark.read.table("workspace.default.optimize_route").toPandas()
    print("✓ optimize_routes loaded")
except Exception as e:
    print(f"✗ Error loading optimize_routes: {e}")
    optimize_df = pd.DataFrame()

# =========================================
# CLEAN TYPES
# =========================================
if not route_df.empty:
    route_df["train_number"] = route_df["trainNumber"].astype(str)

if not optimize_df.empty:
    optimize_df["train_number"] = optimize_df["trainNumber"].astype(str)

# =========================================
# LOAD MODEL (MODEL 1 ONLY)
# =========================================
try:
    with open("(Clone) train_delay_model.pkl", "rb") as f:
        delay_model = pickle.load(f)
    print("✓ Delay model loaded")
except Exception as e:
    print(f"✗ Error loading delay model: {e}")
    delay_model = None

# =========================================
# HELPER: GET DISTANCE + STATION NAME FROM optimize_routes
# =========================================
def get_model_features(train_number, station_name):

    filtered = optimize_df[
        (optimize_df["train_number"] == train_number) &
        (optimize_df["station_name"] == station_name)
    ]

    if filtered.empty:
        return None, None

    distance = filtered.iloc[0]["distance"]
    station = filtered.iloc[0]["toStnCode"]

    return distance, station

# =========================================
# HELPER: DELAY PREDICTION
# =========================================
def predict_delay(distance, month, date):

    if delay_model is None or distance is None:
        return 0.0

    input_data = [[distance, month, date]]
    prediction = delay_model.predict(input_data)

    return float(prediction[0])

# =========================================
# HELPER: SEARCH ROUTES (TABLE 1)
# =========================================
def search_routes(source, destination, min_price, max_price):

    filtered_df = route_df[
        (route_df["source"] == source) &
        (route_df["destination"] == destination)
    ]

    if filtered_df.empty:
        return []

    routes = []

    for _, row in filtered_df.iterrows():

        total_price = row["base_fare"] + row["railway_charges"] + row["tax"]

        if min_price <= total_price <= max_price:
            routes.append(row)

    return routes

# =========================================
# COMBINED API
# =========================================
@app.get("/smart-search")
def smart_search(
    source: str,
    destination: str,
    min_price: float,
    max_price: float,
    month: int,
    date: int
):

    # Validate input
    if min_price > max_price:
        raise HTTPException(status_code=400, detail="Invalid price range")

    if route_df.empty or optimize_df.empty:
        raise HTTPException(status_code=500, detail="Dataset not loaded")

    # Step 1: Get candidate routes
    routes_data = search_routes(source, destination, min_price, max_price)

    if not routes_data:
        raise HTTPException(status_code=404, detail="No routes found")

    routes = []

    # Step 2: Enrich with model 1
    for row in routes_data:

        train_number = row["train_number"]
        station_name = row["station_name"]

        # Get distance from optimize_routes table
        distance, station = get_model_features(train_number, station_name)

        # Predict delay
        delay = predict_delay(distance, month, date)

        total_price = row["base_fare"] + row["railway_charges"] + row["tax"]

        route = {
            "train_number": train_number,
            "station_name": station,
            "departure_time": row["departure_time"],
            "arrival_time": row["arrival_time"],
            "distance": float(distance) if distance else None,
            "total_price": float(total_price),
            "predicted_delay_minutes": delay
        }

        routes.append(route)

    # =====================================
    # RANKING
    # =====================================
    prices = [r["total_price"] for r in routes]
    delays = [r["predicted_delay_minutes"] for r in routes]

    min_price_val, max_price_val = min(prices), max(prices)
    min_delay, max_delay = min(delays), max(delays)

    price_range = max_price_val - min_price_val if max_price_val != min_price_val else 1
    delay_range = max_delay - min_delay if max_delay != min_delay else 1

    PRICE_WEIGHT = 0.6
    DELAY_WEIGHT = 0.4

    for r in routes:
        norm_price = (r["total_price"] - min_price_val) / price_range
        norm_delay = (r["predicted_delay_minutes"] - min_delay) / delay_range

        r["score"] = PRICE_WEIGHT * norm_price + DELAY_WEIGHT * norm_delay

    routes = sorted(routes, key=lambda x: x["score"])

    # Remove score
    for r in routes:
        r.pop("score")

    return {
        "total_routes_found": len(routes),
        "best_routes": routes[:5]
    }

# =========================================
# RUN SERVER
# =========================================
def run():
    uvicorn.run(app, host="0.0.0.0", port=8001)

thread = Thread(target=run)
thread.start()

✓ route_optimisation_data loaded
✓ optimize_routes loaded
✓ Delay model loaded


INFO:     Started server process [6032]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


In [0]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS IN workspace").show()
spark.sql("SHOW TABLES IN workspace.default").show()

+---------+
|  catalog|
+---------+
|  samples|
|   system|
|workspace|
+---------+

+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
+------------------+

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|      optimize_route|      false|
| default|route_optimisatio...|      false|
| default|           schedules|      false|
| default|    train_delay_data|      false|
| default|train_delay_predi...|      false|
+--------+--------------------+-----------+



In [0]:
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-4738689492350064>, line 3
      1 from fastapi.middleware.cors import CORSMiddleware
----> 3 app.add_middleware(
      4     CORSMiddleware,
      5     allow_origins=["*"],
      6     allow_credentials=True,
      7     allow_methods=["*"],
      8     allow_headers=["*"],
      9 )

File /databricks/python/lib/python3.12/site-packages/starlette/applications.py:125, in Starlette.add_middleware(self, middleware_class, *args, **kwargs)
    118 def add_middleware(
    119     self,
    120     middleware_class: _MiddlewareFactory[P],
    121     *args: P.args,
    122     **kwargs: P.kwargs,
    123 ) -> None:
    124     if self.middleware_stack is not None:  # pragma: no cover
--> 125         raise RuntimeError("Cannot add middleware after an application has started")
    126     self.user_middleware.insert(0, Middleware(